# 06 — Prepare Future-Location Scenario

## Research purpose

Combine manually compiled future locations with the current facility baseline using estimated scenario capacities.


## Imports

Load the libraries for data preparation, identifier assignment and spatial grouping.


In [1]:
from collections import defaultdict
import math
from pathlib import Path
import re
import warnings

import pandas as pd

## Project paths

Locate the project folder and define the paths for the manual future locations, current datasets, review records and combined scenario tables.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_MANUAL = PROJECT_ROOT / "data" / "manual"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_FINAL = PROJECT_ROOT / "data" / "final"

MANUAL_INPUT_PATH = DATA_MANUAL / "operator_future_datacenters_manual.csv"
PROCESSED_OUTPUT_PATH = DATA_PROCESSED / "operator_future_datacenters_processed.csv"
REVIEW_OUTPUT_PATH = DATA_INTERIM / "operator_future_datacenters_review.csv"
SCENARIO_OUTPUT_PATH = DATA_FINAL / "hyperscale_facilities_current_plus_future_scenario.csv"
SCENARIO_CAMPUS_OUTPUT_PATH = DATA_FINAL / "hyperscale_campuses_current_plus_future_scenario.csv"

## Scenario assumptions

Set operator PUE values, the allowed future statuses, the capacity-rounding bins and the 1 km campus-grouping threshold.


In [3]:
OPERATOR_PUE = {"Google": 1.09, "Amazon": 1.15, "Microsoft": 1.17}
FALLBACK_PUE = 1.15
ALLOWED_FUTURE_STATUSES = {"planned", "in_development"}
ROUNDING_BINS_MW = [25, 50, 100, 150, 200]
SCENARIO_CAMPUS_DISTANCE_M = 1000.0

## Load future locations and current facilities

Read the manual future-location table and check its identifiers, operators, statuses and coordinate labels before loading the current facility baseline. Use the future coordinates as representative points for renewable-profile matching.


In [4]:
future = pd.read_csv(MANUAL_INPUT_PATH)
required = {
    "future_record_id", "operator_group", "location_name", "country", "country_code",
    "world_region", "sub_region", "latitude", "longitude", "timezone", "status",
    "coordinate_method", "coordinate_confidence", "record_granularity",
}

missing = sorted(required - set(future.columns))
if missing:
    raise ValueError(f"Manual future input is missing {missing}")
if future["future_record_id"].duplicated().any():
    raise ValueError("Duplicate future_record_id")
if not set(future["operator_group"]).issubset({"Amazon", "Google", "Microsoft"}):
    raise ValueError("Unexpected operator")
if not set(future["status"]).issubset(ALLOWED_FUTURE_STATUSES):
    raise ValueError("Unexpected lifecycle status")
for column in ["latitude", "longitude"]:
    future[column] = pd.to_numeric(future[column], errors="coerce")
if future[["latitude", "longitude", "timezone", "coordinate_method", "coordinate_confidence"]].isna().any().any():
    raise ValueError("Manual coordinate provenance is incomplete")

print(future.groupby(["operator_group", "status"]).size().to_string())
current = pd.read_csv(DATA_FINAL / "hyperscale_facilities_final.csv")
print(f"Current facilities: {len(current):,}; future locations: {len(future):,}")

operator_group  status        
Amazon          planned            2
Google          in_development    27
Microsoft       planned            9
Current facilities: 489; future locations: 38


## Load campus capacity benchmarks

Load the current campus table and retain positive capacities as benchmarks for estimating future-location capacities.


In [5]:
benchmarks = pd.read_csv(DATA_FINAL / "hyperscale_campuses_analysis.csv")
required_benchmark_columns = {"operator_group", "sub_region", "world_region", "capacity_mw"}

missing = sorted(required_benchmark_columns - set(benchmarks.columns))
if missing:
    raise ValueError(f"Current campus benchmark is missing {missing}")
benchmarks["capacity_mw"] = pd.to_numeric(benchmarks["capacity_mw"], errors="coerce")
benchmarks = benchmarks[benchmarks["capacity_mw"].gt(0)].copy()

print(f"Positive current-campus benchmarks: {len(benchmarks):,}")
benchmarks[["operator_group", "sub_region", "world_region", "capacity_mw"]].head()

Positive current-campus benchmarks: 164


,operator_group,sub_region,world_region,capacity_mw
0,Amazon,Western Asia,Middle East and Africa,20.370439
1,Amazon,Western Asia,Middle East and Africa,19.208144
2,Amazon,Western Asia,Middle East and Africa,21.107227
3,Amazon,Australia and New Zealand,Asia Pacific,66.249721
4,Amazon,Australia and New Zealand,Asia Pacific,58.351797


## Estimate future capacities

Estimate each future capacity from the first available median in this order: operator and sub-region, operator and world region, all operators in the sub-region, all operators in the world region, then the global sample. Derive IT load using operator PUE and retain the selected method and reference count. Label these scenario estimates as low confidence.


In [6]:
def impute_capacity(row):
    filters = [
        ('estimated_from_current_operator_sub_region_median', (benchmarks['operator_group'] == row['operator_group']) & (benchmarks['sub_region'] == row['sub_region'])),
        ('estimated_from_current_operator_world_region_median', (benchmarks['operator_group'] == row['operator_group']) & (benchmarks['world_region'] == row['world_region'])),
        ('estimated_from_current_all_operator_sub_region_median', benchmarks['sub_region'] == row['sub_region']),
        ('estimated_from_current_all_operator_world_region_median', benchmarks['world_region'] == row['world_region']),
        ('estimated_from_current_global_median', benchmarks['capacity_mw'].notna()),
    ]
    for method, mask in filters:
        subset = benchmarks[mask]
        if subset.empty:
            continue
        capacity_mw = float(subset['capacity_mw'].median())
        pue = OPERATOR_PUE.get(row['operator_group'], FALLBACK_PUE)
        return pd.Series({
            'capacity_mw': capacity_mw, 'it_load_mw': capacity_mw / pue,
            'capacity_confidence': 'low',
            'capacity_imputation_method': method, 'imputation_reference_count': int(len(subset)),
            'imputation_reference_capacity_mw': capacity_mw,
        })
    return pd.Series({
        'capacity_mw': pd.NA, 'it_load_mw': pd.NA,
        'capacity_confidence': 'unknown', 'capacity_imputation_method': 'not_imputed_no_matching_benchmark',
        'imputation_reference_count': 0, 'imputation_reference_capacity_mw': pd.NA,
    })

imputed = future.apply(impute_capacity, axis=1)
for column in imputed.columns:
    future[column] = imputed[column]
for column in ["capacity_mw", "it_load_mw"]:
    if column in future:
        future[column] = pd.to_numeric(future[column], errors="coerce")

future[["future_record_id", "operator_group", "capacity_mw", "it_load_mw", "capacity_imputation_method", "imputation_reference_count"]].head(10)

,future_record_id,operator_group,capacity_mw,it_load_mw,capacity_imputation_method,imputation_reference_count
0,FUTURE-GOOGLE-google-andhra-pradesh-india,Google,47.773908,43.829274,estimated_from_current_operator_world_region_m...,3
1,FUTURE-GOOGLE-google-armstrong-county-texas,Google,105.405027,96.701860,estimated_from_current_operator_sub_region_median,22
2,FUTURE-GOOGLE-google-cedar-rapids-iowa,Google,105.405027,96.701860,estimated_from_current_operator_sub_region_median,22
3,FUTURE-GOOGLE-google-chesterfield-county-virginia,Google,105.405027,96.701860,estimated_from_current_operator_sub_region_median,22
4,FUTURE-GOOGLE-google-chonburi-thailand,Google,45.321195,41.579078,estimated_from_current_operator_sub_region_median,2
5,FUTURE-GOOGLE-google-dietzenbach-germany,Google,191.965167,176.114832,estimated_from_current_operator_sub_region_median,3
6,FUTURE-GOOGLE-google-dorchester-county-south-c...,Google,105.405027,96.701860,estimated_from_current_operator_sub_region_median,22
7,FUTURE-GOOGLE-google-farciennes-belgium,Google,191.965167,176.114832,estimated_from_current_operator_sub_region_median,3
8,FUTURE-GOOGLE-google-groningen-netherlands,Google,191.965167,176.114832,estimated_from_current_operator_sub_region_median,3
9,FUTURE-GOOGLE-google-haskell-county-texas,Google,105.405027,96.701860,estimated_from_current_operator_sub_region_median,22


## Round scenario capacities

Round positive estimates up to the next 25, 50, 100, 150 or 200 MW bin and calculate IT load using operator PUE. Retain the raw estimates and leave values above 200 MW unrounded with a review note. Missing or nonpositive estimates remain unavailable.


In [7]:
def rounded_capacity_bin(value):
    if pd.isna(value):
        return pd.NA
    capacity = float(value)
    if capacity <= 0:
        return pd.NA
    for bin_mw in ROUNDING_BINS_MW:
        if capacity <= bin_mw:
            return float(bin_mw)
    return capacity

future["review_notes"] = ""
future["capacity_mw"] = pd.to_numeric(future["capacity_mw"], errors="coerce")
future["it_load_mw"] = pd.to_numeric(future["it_load_mw"], errors="coerce")
future["assumed_pue"] = future["operator_group"].map(OPERATOR_PUE).fillna(FALLBACK_PUE)
future["scenario_capacity_mw_rounded"] = future["capacity_mw"].apply(rounded_capacity_bin)
future["scenario_it_load_mw_rounded"] = (
    future["scenario_capacity_mw_rounded"] / future["assumed_pue"]
)
future["scenario_capacity_rounding_method"] = "round_up_to_next_25_50_100_150_200_mw_bin"

above_bins = future["capacity_mw"] > max(ROUNDING_BINS_MW)
if above_bins.any():
    future.loc[above_bins, "scenario_capacity_rounding_method"] = (
        "not_rounded_capacity_above_200_mw_bin"
    )
    future.loc[above_bins, "review_notes"] = "capacity_rounding_bin_exceeded;"

future[["future_record_id", "capacity_mw", "scenario_capacity_mw_rounded", "scenario_it_load_mw_rounded", "review_notes"]].head(10)

,future_record_id,capacity_mw,scenario_capacity_mw_rounded,scenario_it_load_mw_rounded,review_notes
0,FUTURE-GOOGLE-google-andhra-pradesh-india,47.773908,50.0,45.871560,
1,FUTURE-GOOGLE-google-armstrong-county-texas,105.405027,150.0,137.614679,
2,FUTURE-GOOGLE-google-cedar-rapids-iowa,105.405027,150.0,137.614679,
3,FUTURE-GOOGLE-google-chesterfield-county-virginia,105.405027,150.0,137.614679,
4,FUTURE-GOOGLE-google-chonburi-thailand,45.321195,50.0,45.871560,
5,FUTURE-GOOGLE-google-dietzenbach-germany,191.965167,200.0,183.486239,
6,FUTURE-GOOGLE-google-dorchester-county-south-c...,105.405027,150.0,137.614679,
7,FUTURE-GOOGLE-google-farciennes-belgium,191.965167,200.0,183.486239,
8,FUTURE-GOOGLE-google-groningen-netherlands,191.965167,200.0,183.486239,
9,FUTURE-GOOGLE-google-haskell-county-texas,105.405027,150.0,137.614679,


## Assign future facility identifiers

Extend each existing country/operator identifier sequence in a fixed location and future-record order. Sort the future records and collect those with capacity review notes.


In [8]:
# Normalize an operator name for its facility-ID token.
def operator_id_token(value) -> str:
    token = re.sub(r"[^A-Z0-9]+", "", str(value or "").upper())
    return token or "UNKNOWN"


# Continue each country/operator facility sequence in deterministic future-record order.
def assign_future_facility_ids(current: pd.DataFrame, future: pd.DataFrame) -> pd.DataFrame:
    """Assign future IDs after the current max for each country/operator pair."""
    if future.empty:
        return future

    future = future.copy()
    current_max = {}
    pattern = re.compile(r"^(?P<country>[A-Z]{3})-(?P<operator>[A-Z0-9]+)-(?P<number>\d+)$")

    for _, row in current.iterrows():
        facility_id = str(row.get("facility_id", ""))
        match = pattern.match(facility_id)
        country = str(row.get("country_code") or (match.group("country") if match else "XXX")).upper()
        operator = operator_id_token(row.get("operator_group") or (match.group("operator") if match else "UNKNOWN"))
        suffix = int(match.group("number")) if match else 0
        key = (country, operator)
        current_max[key] = max(current_max.get(key, 0), suffix)

    next_suffix = current_max.copy()
    future_ids = pd.Series(pd.NA, index=future.index, dtype="object")
    sort_columns = ["country_code", "operator_group", "location_name", "future_record_id"]
    for idx, row in future.sort_values(sort_columns, na_position="last").iterrows():
        country = str(row.get("country_code") or "XXX").upper()
        operator = operator_id_token(row.get("operator_group"))
        key = (country, operator)
        next_suffix[key] = next_suffix.get(key, 0) + 1
        future_ids.loc[idx] = f"{country}-{operator}-{next_suffix[key]:04d}"

    future["facility_id"] = future_ids
    return future

future = assign_future_facility_ids(current, future)
future = future.sort_values(["operator_group", "country_code", "location_name"], na_position="last").reset_index(drop=True)
review = future[future["review_notes"].ne("")].copy()

future[["future_record_id", "facility_id", "operator_group", "location_name"]].head(10)

,future_record_id,facility_id,operator_group,location_name
0,FUTURE-AMAZON-aws-chile,CHL-AMAZON-0005,Amazon,Chile
1,FUTURE-AMAZON-aws-saudi-arabia,SAU-AMAZON-0001,Amazon,Saudi Arabia
2,FUTURE-GOOGLE-google-kronstorf-austria,AUT-GOOGLE-0001,Google,"Kronstorf, Austria"
3,FUTURE-GOOGLE-google-farciennes-belgium,BEL-GOOGLE-0002,Google,"Farciennes, Belgium"
4,FUTURE-GOOGLE-google-dietzenbach-germany,DEU-GOOGLE-0001,Google,"Dietzenbach, Germany"
5,FUTURE-GOOGLE-google-andhra-pradesh-india,IND-GOOGLE-0001,Google,"Andhra Pradesh, India"
6,FUTURE-GOOGLE-google-selangor-malaysia,MYS-GOOGLE-0003,Google,"Selangor, Malaysia"
7,FUTURE-GOOGLE-google-groningen-netherlands,NLD-GOOGLE-0003,Google,"Groningen, Netherlands"
8,FUTURE-GOOGLE-google-skien-norway,NOR-GOOGLE-0001,Google,"Skien, Norway"
9,FUTURE-GOOGLE-google-horndal-sweden,SWE-GOOGLE-0001,Google,"Horndal, Sweden"


## Prepare current facility records

Select the current facility fields for the combined scenario and retain their capacities, identifiers, source-campus membership and notes. Label their coordinates as OSM geometries or centroids.


In [9]:
current_layer = pd.DataFrame({
    "scenario_record_id": "CURRENT-" + current["facility_id"].astype("string"),
    "record_layer": "current_operational_osm",
    "facility_id": current["facility_id"],
    "future_record_id": pd.NA,
    "operator_group": current["operator_group"],
    "location_name": current["facility_name"],
    "country": current["country"],
    "country_code": current["country_code"],
    "world_region": current["world_region"],
    "sub_region": current["sub_region"],
    "timezone": current["timezone"],
    "latitude": pd.to_numeric(current["latitude"], errors="coerce"),
    "longitude": pd.to_numeric(current["longitude"], errors="coerce"),
    "status": current["status"],
    "capacity_mw": pd.to_numeric(current["capacity_mw"], errors="coerce"),
    "it_load_mw": pd.to_numeric(current["it_load_mw"], errors="coerce"),
    "raw_imputed_capacity_mw": pd.NA,
    "raw_imputed_it_load_mw": pd.NA,
    "capacity_confidence": current["capacity_confidence"],
    "source_type": current["source_type"],
    "coordinate_method": "osm_geometry_or_centroid",
    "coordinate_confidence": "medium",
    "coordinate_notes": "OSM representative coordinate or geometry centroid for current operational baseline.",
    "record_granularity": "osm_facility_or_campus",
    "source_campus_group_id": current.get("campus_group_id", pd.NA),
    "notes": current["notes"],
})

current_layer[["facility_id", "operator_group", "location_name", "capacity_mw", "source_campus_group_id"]].head()

,facility_id,operator_group,location_name,capacity_mw,source_campus_group_id
0,USA-GOOGLE-0061,Google,Google Fiber,46.775393,CAMPUS-00001
1,SGP-AMAZON-0002,Amazon,Amazon SIN4,61.643197,CAMPUS-00002
2,MYS-GOOGLE-0001,Google,Google Singapore DC,42.868482,CAMPUS-00004
3,SGP-AMAZON-0001,Amazon,Amazon SIN3,39.755464,CAMPUS-00005
4,SGP-AMAZON-0003,Amazon,Amazon SIN5,30.783234,CAMPUS-00002


## Combine current and future records

Use rounded capacities for the future scenario and retain their raw estimates and manual coordinate labels. Combine the current and future records, keeping fields that apply to only one group blank in the other.


In [10]:
future_layer = pd.DataFrame({
    "scenario_record_id": future["future_record_id"],
    "record_layer": "future_operator_source",
    "facility_id": future["facility_id"],
    "future_record_id": future["future_record_id"],
    "operator_group": future["operator_group"],
    "location_name": future["location_name"],
    "country": future["country"],
    "country_code": future["country_code"],
    "world_region": future["world_region"],
    "sub_region": future["sub_region"],
    "timezone": future["timezone"],
    "latitude": pd.to_numeric(future["latitude"], errors="coerce"),
    "longitude": pd.to_numeric(future["longitude"], errors="coerce"),
    "status": future["status"],
    "capacity_mw": pd.to_numeric(future["scenario_capacity_mw_rounded"], errors="coerce"),
    "it_load_mw": pd.to_numeric(future["scenario_it_load_mw_rounded"], errors="coerce"),
    "raw_imputed_capacity_mw": pd.to_numeric(future["capacity_mw"], errors="coerce"),
    "raw_imputed_it_load_mw": pd.to_numeric(future["it_load_mw"], errors="coerce"),
    "capacity_confidence": future["capacity_confidence"],
    "source_type": "official_operator_future_source",
    "coordinate_method": future["coordinate_method"],
    "coordinate_confidence": future["coordinate_confidence"],
    "coordinate_notes": pd.NA,
    "record_granularity": future["record_granularity"],
    "source_campus_group_id": pd.NA,
    "notes": pd.NA,
})

current_layer["future_record_id"] = current_layer["future_record_id"].astype("string")
for column in ["raw_imputed_capacity_mw", "raw_imputed_it_load_mw"]:
    current_layer[column] = current_layer[column].astype("Float64")
future_layer["source_campus_group_id"] = future_layer["source_campus_group_id"].astype("string")
# Deliberately blank cross-layer provenance fields trigger a pandas dtype warning.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.",
        category=FutureWarning,
    )
    scenario = pd.concat([current_layer, future_layer], ignore_index=True, sort=False)

scenario.groupby(["record_layer", "operator_group"], dropna=False)["capacity_mw"].agg(["count", "sum"])

count          sum
record_layer            operator_group                    
current_operational_osm Amazon            298  9923.727280
                        Google             82  4567.907249
                        Microsoft         109  4927.283747
future_operator_source  Amazon              2    50.000000
                        Google             27  3850.000000
                        Microsoft           9   900.000000

## Define campus connectivity

Define a union-find structure to retain transitive campus membership and a great-circle distance function for the 1 km grouping rule. Define helpers for retaining source identifiers and text values within each group.


In [11]:
class UnionFind:
    """Keep transitive campus membership without collapsing facility records."""

    # Start each record in its own connected component.
    def __init__(self, values):
        self.parent = {value: value for value in values}

    # Find a component representative and shorten its parent chain.
    def find(self, value):
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = self.parent[value]
        return value

    # Join connected components without removing their records.
    def union(self, left, right):
        left, right = self.find(left), self.find(right)
        if left != right:
            self.parent[right] = left

    # Collect record identifiers by their transitive component.
    def groups(self):
        grouped = defaultdict(list)
        for value in self.parent:
            grouped[self.find(value)].append(value)
        return list(grouped.values())


# Calculate great-circle separation in metres.
def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle separation of representative coordinates, in metres."""
    radius = 6371008.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi, dlambda = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*radius*math.asin(math.sqrt(a))


def first_non_null(series):
    values = series.dropna()
    if values.empty:
        return pd.NA
    values = values.astype("string").str.strip()
    values = values[values.ne("")]
    return values.iloc[0] if not values.empty else pd.NA

def compact_unique(series, separator="|"):
    values = series.dropna().astype("string").str.strip()
    values = values[values.ne("")]
    return separator.join(sorted(values.unique()))

## Group scenario campuses

Group locations from the same operator within 1 km and retain links within existing current campuses. Reuse an existing campus identifier when a group has exactly one source-campus identifier and assign new identifiers otherwise. Keep records without usable coordinates as separate groups.


In [12]:
def assign_scenario_campus_groups(scenario: pd.DataFrame) -> pd.DataFrame:
    """Assign non-collapsing campus groups using current campus IDs plus the 1000 m rule."""
    if scenario.empty:
        return scenario

    scenario = scenario.copy()
    for column in ["latitude", "longitude"]:
        scenario[column] = pd.to_numeric(scenario[column], errors="coerce")

    scenario["campus_group_id"] = pd.NA
    scenario["campus_group_size"] = 1
    scenario["campus_group_facility_ids"] = scenario["facility_id"].fillna("")
    scenario["campus_group_source_ids"] = scenario.get("source_campus_group_id", pd.NA)

    valid = scenario[scenario["latitude"].notna() & scenario["longitude"].notna()].copy()
    if valid.empty:
        scenario["campus_group_id"] = [f"SCENARIO-CAMPUS-{i:05d}" for i in range(1, len(scenario) + 1)]
        return scenario

    ids = list(valid.index)
    uf = UnionFind(ids)
    for i, left_id in enumerate(ids):
        left = scenario.loc[left_id]
        for right_id in ids[i + 1 :]:
            right = scenario.loc[right_id]
            if left["operator_group"] != right["operator_group"]:
                continue
            distance = haversine_m(
                float(left["latitude"]),
                float(left["longitude"]),
                float(right["latitude"]),
                float(right["longitude"]),
            )
            if distance <= SCENARIO_CAMPUS_DISTANCE_M:
                uf.union(left_id, right_id)

    if "source_campus_group_id" in scenario.columns:
        source_groups = scenario.loc[valid.index].copy()
        source_groups["source_campus_group_id"] = source_groups["source_campus_group_id"].astype("string").str.strip()
        source_groups = source_groups[source_groups["source_campus_group_id"].notna() & source_groups["source_campus_group_id"].ne("")]
        for _, source_group in source_groups.groupby("source_campus_group_id", sort=False):
            group_indices = list(source_group.index)
            for idx in group_indices[1:]:
                uf.union(group_indices[0], idx)

    groups = uf.groups()
    ordered_groups = sorted(
        groups,
        key=lambda indices: (
            str(first_non_null(scenario.loc[indices, "operator_group"])),
            str(first_non_null(scenario.loc[indices, "country_code"])),
            float(pd.to_numeric(scenario.loc[indices, "latitude"], errors="coerce").mean()),
            float(pd.to_numeric(scenario.loc[indices, "longitude"], errors="coerce").mean()),
            str(first_non_null(scenario.loc[indices, "facility_id"])),
        ),
    )

    assigned = set()
    for number, group_indices in enumerate(ordered_groups, start=1):
        group_df = scenario.loc[group_indices].copy()
        source_campus_ids = compact_unique(group_df.get("source_campus_group_id", pd.Series(dtype="object")))
        source_id_values = [value for value in source_campus_ids.split("|") if value]
        campus_id = source_id_values[0] if len(source_id_values) == 1 else f"SCENARIO-CAMPUS-{number:05d}"
        facility_ids = compact_unique(group_df["facility_id"])
        scenario.loc[group_indices, "campus_group_id"] = campus_id
        scenario.loc[group_indices, "campus_group_size"] = len(group_df)
        scenario.loc[group_indices, "campus_group_facility_ids"] = facility_ids
        scenario.loc[group_indices, "campus_group_source_ids"] = source_campus_ids
        assigned.update(group_indices)

    missing_indices = [idx for idx in scenario.index if idx not in assigned]
    start = len(ordered_groups) + 1
    for offset, idx in enumerate(missing_indices):
        campus_id = f"SCENARIO-CAMPUS-{start + offset:05d}"
        scenario.loc[idx, "campus_group_id"] = campus_id
        scenario.loc[idx, "campus_group_size"] = 1
        scenario.loc[idx, "campus_group_facility_ids"] = scenario.loc[idx, "facility_id"]
        scenario.loc[idx, "campus_group_source_ids"] = scenario.loc[idx, "source_campus_group_id"] if "source_campus_group_id" in scenario.columns else pd.NA

    return scenario

scenario = assign_scenario_campus_groups(scenario)
scenario = scenario.sort_values(
    ["operator_group", "country_code", "campus_group_id", "record_layer", "location_name"],
    na_position="last",
).reset_index(drop=True)

scenario[["facility_id", "operator_group", "campus_group_id", "campus_group_size", "capacity_mw"]].head(10)

,facility_id,operator_group,campus_group_id,campus_group_size,capacity_mw
0,ARE-AMAZON-0001,Amazon,CAMPUS-00107,1,20.370439
1,ARE-AMAZON-0002,Amazon,CAMPUS-00161,1,19.208144
2,ARE-AMAZON-0003,Amazon,CAMPUS-00162,1,21.107227
3,AUS-AMAZON-0001,Amazon,CAMPUS-00068,2,48.353052
4,AUS-AMAZON-0003,Amazon,CAMPUS-00068,2,17.896670
5,AUS-AMAZON-0002,Amazon,CAMPUS-00094,2,44.844273
6,AUS-AMAZON-0005,Amazon,CAMPUS-00094,2,13.507524
7,AUS-AMAZON-0004,Amazon,CAMPUS-00140,1,19.558810
8,AUS-AMAZON-0006,Amazon,CAMPUS-00141,1,10.984119
9,AUS-AMAZON-0007,Amazon,CAMPUS-00155,1,9.643418


## Define campus summary calculations

Define helpers for capacity totals, status summaries and representative coordinates weighted by positive capacity. Leave entirely missing totals undefined and use mean coordinates when positive weights are unavailable. Calculate the greatest distance from a campus record to its representative point.


In [13]:
def compact_status(series):
    values = series.dropna().astype("string").str.strip()
    values = values[values.ne("")]
    if values.empty:
        return pd.NA
    unique = sorted(values.unique())
    return unique[0] if len(unique) == 1 else "mixed"

def safe_sum(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return values.sum() if not values.empty else pd.NA

def mean_numeric(series):
    values = pd.to_numeric(series, errors="coerce")
    return values.mean() if values.notna().any() else pd.NA

def weighted_mean(values, weights):
    numeric_values = pd.to_numeric(values, errors="coerce")
    numeric_weights = pd.to_numeric(weights, errors="coerce").fillna(0)
    valid = numeric_values.notna() & numeric_weights.gt(0)
    if valid.any():
        return float((numeric_values[valid] * numeric_weights[valid]).sum() / numeric_weights[valid].sum())
    return mean_numeric(values)

def max_distance_to_point_m(group, lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return pd.NA
    distances = []
    for _, row in group.iterrows():
        if pd.isna(row.get("latitude")) or pd.isna(row.get("longitude")):
            continue
        distances.append(haversine_m(float(lat), float(lon), float(row["latitude"]), float(row["longitude"])))
    return max(distances) if distances else pd.NA

## Aggregate scenario campuses

Create one summary per campus with capacity totals, representative coordinates, current and future record counts, and source membership. Retain raw capacity estimates and coordinate evidence in the combined table.


In [14]:
if scenario.empty:
    scenario_campuses = pd.DataFrame()
else:
    work = scenario.copy()
    if "campus_group_id" not in work.columns or work["campus_group_id"].isna().any():
        work = assign_scenario_campus_groups(work)

    records = []
    for campus_id, group in work.groupby("campus_group_id", dropna=False, sort=True):
        capacity_mw = safe_sum(group["capacity_mw"])
        it_load_mw = safe_sum(group["it_load_mw"])
        campus_lat = weighted_mean(group["latitude"], group["capacity_mw"])
        campus_lon = weighted_mean(group["longitude"], group["capacity_mw"])
        record_layers = set(group["record_layer"].dropna().astype(str))
        records.append({
            "campus_group_id": campus_id,
            "operator_group": first_non_null(group["operator_group"]),
            "country": first_non_null(group["country"]),
            "country_code": first_non_null(group["country_code"]),
            "world_region": first_non_null(group["world_region"]),
            "sub_region": first_non_null(group["sub_region"]),
            "timezone": first_non_null(group["timezone"]),
            "latitude": campus_lat,
            "longitude": campus_lon,
            "facility_count": int(len(group)),
            "current_facility_count": int(group["record_layer"].eq("current_operational_osm").sum()),
            "future_facility_count": int(group["record_layer"].eq("future_operator_source").sum()),
            "record_layer": "current_and_future" if len(record_layers) > 1 else first_non_null(group["record_layer"]),
            "status": compact_status(group["status"]),
            "capacity_mw": capacity_mw,
            "it_load_mw": it_load_mw,
            "raw_imputed_capacity_mw": safe_sum(group["raw_imputed_capacity_mw"]),
            "raw_imputed_it_load_mw": safe_sum(group["raw_imputed_it_load_mw"]),
            "facility_ids": compact_unique(group["facility_id"]),
            "future_record_ids": compact_unique(group["future_record_id"]),
            "source_campus_group_ids": compact_unique(group.get("source_campus_group_id", pd.Series(dtype="object"))),
            "source_types": compact_unique(group["source_type"]),
            "coordinate_methods": compact_unique(group["coordinate_method"]),
            "coordinate_confidence": compact_status(group["coordinate_confidence"]),
            "max_distance_to_representative_m": max_distance_to_point_m(group, campus_lat, campus_lon),
            "notes": (
                "Scenario campus aggregated from current and future facility rows using the "
                f"same-operator within {int(SCENARIO_CAMPUS_DISTANCE_M)} m non-collapsing rule."
            ),
        })

    scenario_campuses = pd.DataFrame(records)
    scenario_campuses = scenario_campuses.sort_values(
        ["operator_group", "country_code", "campus_group_id"],
        na_position="last",
    ).reset_index(drop=True)

scenario_campuses.head(10)

,campus_group_id,operator_group,country,country_code,world_region,sub_region,timezone,latitude,longitude,facility_count,...,raw_imputed_capacity_mw,raw_imputed_it_load_mw,facility_ids,future_record_ids,source_campus_group_ids,source_types,coordinate_methods,coordinate_confidence,max_distance_to_representative_m,notes
0,CAMPUS-00107,Amazon,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,24.893127,55.109898,1,...,<NA>,<NA>,ARE-AMAZON-0001,,CAMPUS-00107,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
1,CAMPUS-00161,Amazon,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,24.268809,54.463322,1,...,<NA>,<NA>,ARE-AMAZON-0002,,CAMPUS-00161,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
2,CAMPUS-00162,Amazon,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,24.708930,54.741704,1,...,<NA>,<NA>,ARE-AMAZON-0003,,CAMPUS-00162,osm,osm_geometry_or_centroid,medium,7.177505e-10,Scenario campus aggregated from current and fu...
3,CAMPUS-00068,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,-34.036750,150.767560,2,...,<NA>,<NA>,AUS-AMAZON-0001|AUS-AMAZON-0003,,CAMPUS-00068,osm,osm_geometry_or_centroid,medium,1.895360e+02,Scenario campus aggregated from current and fu...
4,CAMPUS-00094,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,-33.794864,150.869664,2,...,<NA>,<NA>,AUS-AMAZON-0002|AUS-AMAZON-0005,,CAMPUS-00094,osm,osm_geometry_or_centroid,medium,7.286535e+01,Scenario campus aggregated from current and fu...
5,CAMPUS-00140,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,-33.809560,150.839035,1,...,<NA>,<NA>,AUS-AMAZON-0004,,CAMPUS-00140,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
6,CAMPUS-00141,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,-33.780105,151.125879,1,...,<NA>,<NA>,AUS-AMAZON-0006,,CAMPUS-00141,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
7,CAMPUS-00155,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Melbourne,-37.838017,144.755673,1,...,<NA>,<NA>,AUS-AMAZON-0007,,CAMPUS-00155,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
8,CAMPUS-00092,Amazon,Bahrain,BHR,Middle East and Africa,Western Asia,Asia/Bahrain,26.154517,50.474019,1,...,<NA>,<NA>,BHR-AMAZON-0001,,CAMPUS-00092,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
9,CAMPUS-00150,Amazon,Bahrain,BHR,Middle East and Africa,Western Asia,Asia/Bahrain,26.050534,50.503772,1,...,<NA>,<NA>,BHR-AMAZON-0002,,CAMPUS-00150,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...


## Reconcile scenario totals

Check that the combined scenario contains every current and future record exactly once. Verify that current capacity plus rounded future capacity equals the totals in both the facility and campus tables.


In [15]:
current_total = current["capacity_mw"].sum()
future_total = future["scenario_capacity_mw_rounded"].sum()
scenario_total = scenario["capacity_mw"].sum()
campus_total = scenario_campuses["capacity_mw"].sum()
if len(scenario) != len(current) + len(future):
    raise ValueError("Combined scenario does not contain every current and future record")
if scenario["scenario_record_id"].duplicated().any() or scenario["facility_id"].duplicated().any():
    raise ValueError("Combined scenario identifiers are not unique")
if not math.isclose(scenario_total, current_total + future_total, rel_tol=1e-12, abs_tol=1e-9):
    raise ValueError("Combined capacity does not equal current plus rounded future capacity")
if not math.isclose(campus_total, scenario_total, rel_tol=1e-12, abs_tol=1e-9):
    raise ValueError("Scenario facility and campus capacities do not reconcile")


print(f"Future rows: {len(future):,}; combined facilities: {len(scenario):,}; campuses: {len(scenario_campuses):,}")
pd.DataFrame({
    "dataset": ["current facilities", "rounded future locations", "combined facilities", "combined campuses"],
    "capacity_mw": [current_total, future_total, scenario_total, campus_total],
})

Future rows: 38; combined facilities: 527; campuses: 202


,dataset,capacity_mw
0,current facilities,19418.918276
1,rounded future locations,4800.000000
2,combined facilities,24218.918276
3,combined campuses,24218.918276


## Save scenario tables

Save the processed future estimates, records needing review and the combined facility and campus scenario tables.


In [16]:
future.to_csv(PROCESSED_OUTPUT_PATH, index=False)
review.to_csv(REVIEW_OUTPUT_PATH, index=False)
scenario.to_csv(SCENARIO_OUTPUT_PATH, index=False)
scenario_campuses.to_csv(SCENARIO_CAMPUS_OUTPUT_PATH, index=False)

print(f"Future rows: {len(future):,}; combined facilities: {len(scenario):,}; campuses: {len(scenario_campuses):,}")

Future rows: 38; combined facilities: 527; campuses: 202
